# جلسه ۲: مهندسی پرامپت در عمل

## اهداف
- تسلط بر پرامپت‌نویسی zero-shot، few-shot و chain-of-thought
- طراحی پرامپت‌های سیستم مؤثر
- استفاده از قالب‌های پرامپت برای الگوهای قابل استفاده مجدد
- مقایسه استراتژی‌های پرامپت‌نویسی در وظایف واقعی

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv()

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "rnj-1-instruct")

def chat(user_message, system_message="You are a helpful assistant.", model=MODEL, temperature=0.7):
    """تابع کمکی ساده برای فراخوانی Chat Completions API."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. پرامپت‌نویسی Zero-Shot

**Zero-shot** = دادن وظیفه به مدل بدون هیچ مثالی.
مدل کاملاً بر دانش آموزشی خود تکیه می‌کند.

برای وظایف ساده و خوش‌تعریف خوب کار می‌کند.

In [ ]:
# Zero-shot: طبقه‌بندی احساسات بدون هیچ مثالی
prompt = """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The food was absolutely delicious and the service was impeccable!"

Sentiment:"""

result = chat(prompt, temperature=0)
print(result)

POSITIVE


In [ ]:
# Zero-shot: استخراج اطلاعات
prompt = """Extract the person's name, age, and occupation from this text.

Text: "Dr. Sarah Chen, a 42-year-old neuroscientist at MIT, published groundbreaking research."

Output:"""

result = chat(prompt, temperature=0)
print(result)

```json
{"name":"Dr. Sarah Chen","age":42,"occupation":"neuroscientist at MIT"}
```


## ۲. پرامپت‌نویسی Few-Shot

**Few-shot** = ارائه چند مثال قبل از وظیفه اصلی.
این به مدل کمک می‌کند تا درک کند:
- قالب دقیقی که می‌خواهید
- موارد لبه‌ای و رفتار مورد انتظار
- «سبک» خروجی

In [ ]:
# Few-shot: طبقه‌بندی احساسات با مثال‌ها
prompt = """Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "Great product, works perfectly!"
Sentiment: POSITIVE

Review: "Terrible quality, broke after one day."
Sentiment: NEGATIVE

Review: "It's okay, nothing special."
Sentiment: NEUTRAL

Review: "The battery life could be better but the camera is amazing."
Sentiment:"""

result = chat(prompt, temperature=0)
print(result)

Sentiment: NEUTRAL


In [ ]:
# Few-shot: طبقه‌بندی متن سفارشی
# مثال‌ها دسته‌بندی‌های خاص شما را به مدل آموزش می‌دهند
prompt = """Classify the support ticket into one of these categories: BILLING, TECHNICAL, ACCOUNT, OTHER.

Ticket: "I was charged twice for my subscription"
Category: BILLING

Ticket: "The app keeps crashing on my phone"
Category: TECHNICAL

Ticket: "I need to reset my password"
Category: ACCOUNT

Ticket: "My dashboard is not loading and shows a 500 error"
Category:"""

result = chat(prompt, temperature=0)
print(result)

**Category:** TECHNICAL


## ۳. پرامپت‌نویسی Chain-of-Thought (CoT)

**Chain-of-Thought** = از مدل بخواهید قبل از پاسخ‌دهی «قدم به قدم فکر کند».

این عملکرد را در موارد زیر به شدت بهبود می‌بخشد:
- مسائل ریاضی و منطقی
- وظایف استدلال پیچیده
- تحلیل چندمرحله‌ای

دو رویکرد:
1. **CoT ساده**: اضافه کردن «قدم به قدم فکر کن» به پرامپت
2. **CoT با مثال (Few-shot CoT)**: نمایش مثال‌ها با مراحل استدلال

In [ ]:
# بدون CoT — مدل ممکن است اشتباه جواب دهد
prompt_no_cot = """A store has 15 apples. It sells 8 in the morning and receives a shipment of 12.
Then it sells 6 more in the afternoon. How many apples are left?

Answer:"""

print("بدون CoT:")
print(chat(prompt_no_cot, temperature=0))

# با CoT — از مدل می‌خواهیم قدم به قدم استدلال کند
prompt_cot = """A store has 15 apples. It sells 8 in the morning and receives a shipment of 12.
Then it sells 6 more in the afternoon. How many apples are left?

Think step by step, then give the final answer."""

print("\nبا CoT:")
print(chat(prompt_cot, temperature=0))

Without CoT:


$15 - 8 + 12 - 6 = 13$

Answer: 13 apples.

With CoT:


Step 1: Start with $15$ apples.  
Step 2: Sells $8$: $15 - 8 = 7$.  
Step 3: Receives $12$: $7 + 12 = 19$.  
Step 4: Sells $6$: $19 - 6 = 13$.

Final answer: $13$ apples.


In [ ]:
# CoT با مثال: نمایش فرآیند استدلال در مثال‌ها
prompt = """Determine if the conclusion follows from the premise. Show your reasoning.

Premise: "All mammals are warm-blooded. Whales are mammals."
Reasoning: Since all mammals are warm-blooded and whales are mammals, whales must be warm-blooded.
Conclusion follows: Yes

Premise: "Some birds can fly. Penguins are birds."
Reasoning: The premise says SOME birds can fly, not ALL. Penguins being birds doesn't guarantee they can fly.
Conclusion follows: No

Premise: "All students who passed the exam studied hard. John studied hard."
Reasoning:"""

result = chat(prompt, temperature=0)
print(result)

Reasoning: The premise says "If a student passed, then they studied hard" (passed → studied). From "John studied hard" (studied) you cannot infer he passed — this is affirming the consequent. A counterexample: a student could study hard and still fail.

Conclusion follows: No


## ۴. الگوهای طراحی پرامپت سیستم

**پرامپت سیستم** قدرتمندترین ابزار شماست. الگوهای رایج:

1. **تخصیص نقش**: "شما یک [نقش] هستید..."
2. **محدودیت‌ها**: "فقط با... پاسخ دهید"، "هرگز..."
3. **مشخصات قالب**: "با نقاط بولت پاسخ دهید"
4. **قوانین رفتاری**: "اگر نمی‌دانید، بگویید"

In [ ]:
# الگو ۱: نقش متخصص با محدودیت‌ها
system_prompt = """You are a senior Python code reviewer.
- Review the code for bugs, style issues, and improvements
- Rate the code quality: GOOD, NEEDS_IMPROVEMENT, or POOR
- Keep feedback concise (max 3 bullet points)
- Always suggest at least one improvement"""

code_to_review = """
def calc(x,y,op):
    if op == 'add': return x+y
    if op == 'sub': return x-y
    if op == 'mul': return x*y
    if op == 'div': return x/y
"""

result = chat(code_to_review, system_message=system_prompt, temperature=0)
print(result)

- Rating: NEEDS_IMPROVEMENT
- Issues:
  1. No error handling (division by zero, unknown op) and no type hints/docstring.  
  2. Repetitive ifs — harder to extend; uses string literals for ops which is error-prone.
- Suggested improvement: use a mapping to functions, add type hints, explicit errors.

```python
# python
from typing import Callable
import operator

def calc(x: float, y: float, op: str) -> float:
    """Simple calculator: op in {'add','sub','mul','div'}."""
    ops: dict[str, Callable[[float, float], float]] = {
        "add": operator.add,
        "sub": operator.sub,
        "mul": operator.mul,
        "div": operator.truediv,
    }
    fn = ops.get(op)
    if fn is None:
        raise ValueError(f"unknown op: {op!r}")
    if op == "div" and y == 0:
        raise ZeroDivisionError("division by zero")
    return fn(x, y)
```


In [ ]:
# الگو ۲: دستورالعمل خروجی ساختاریافته
system_prompt = """You are a text analysis assistant.
For every input text, respond with EXACTLY this format:

TOPIC: [main topic]
TONE: [formal/informal/neutral]
KEY_POINTS: [comma-separated key points]
WORD_COUNT: [approximate word count of input]"""

text = """Artificial intelligence has been transforming industries at an unprecedented rate.
From healthcare to finance, AI-powered solutions are improving efficiency and accuracy.
However, ethical concerns around bias and privacy remain significant challenges."""

result = chat(text, system_message=system_prompt, temperature=0)
print(result)

TOPIC: Artificial intelligence adoption and challenges
TONE: neutral
KEY_POINTS: AI transforming industries, applications in healthcare and finance, AI improves efficiency and accuracy, ethical concerns about bias and privacy
WORD_COUNT: 31


## ۵. قالب‌های پرامپت

از f-strings پایتون برای ساخت قالب‌های پرامپت قابل استفاده مجدد استفاده کنید.
این پرامپت‌های شما را قابل نگهداری و پارامتریک می‌کند.

In [ ]:
# قالب پرامپت قابل استفاده مجدد برای وظایف مختلف
def analyze_text(text, analysis_type):
    """تحلیل متن با نوع تحلیل مشخص‌شده."""
    template = f"""Perform {analysis_type} analysis on the following text.
Be concise and specific in your analysis.

Text: \"{text}\"

Analysis:"""
    return chat(template, temperature=0)

sample_text = "The new policy will increase taxes for high earners while providing relief for small businesses."

# استفاده از همان قالب برای انواع مختلف تحلیل
print("=== تحلیل احساسات ===")
print(analyze_text(sample_text, "sentiment"))

print("\n=== تحلیل سوگیری ===")
print(analyze_text(sample_text, "bias"))

=== Sentiment Analysis ===


- Overall sentiment: Mixed/Neutral — balances negative and positive impacts.
- For high earners: Negative (policy increases taxes).
- For small businesses: Positive (policy provides relief).
- Tone: Informative/objective; no emotional language.

=== Bias Analysis ===


## تمرین: مقایسه استراتژی‌های پرامپت‌نویسی

یک طبقه‌بندی‌کننده بررسی محصول بسازید که بررسی‌ها را دسته‌بندی و اطلاعات کلیدی را استخراج کند.
رویکردهای **zero-shot**، **few-shot** و **CoT** را امتحان کنید و نتایج را مقایسه کنید.

In [ ]:
# بررسی تست برای همه استراتژی‌ها
test_review = """The laptop arrived quickly but the packaging was damaged.
The device itself works fine — fast processor, beautiful display.
However, the battery barely lasts 3 hours which is disappointing for the price.
Customer support was helpful when I reported the packaging issue."""

# استراتژی ۱: Zero-shot
zero_shot_prompt = f"""Analyze this product review. Provide: overall sentiment, 
pros, cons, and a rating out of 5.

Review: \"{test_review}\""""

print("=== Zero-Shot ===")
print(chat(zero_shot_prompt, temperature=0))

=== Zero-Shot ===


- Overall sentiment: Mixed (positive about performance and support, negative about packaging and battery life)

- Pros:
  - Fast processor
  - Beautiful display
  - Quick delivery
  - Helpful customer support

- Cons:
  - Damaged packaging on arrival
  - Battery only ~3 hours, disappointing for the price

- Rating: 3/5


In [ ]:
# استراتژی ۲: Few-shot
few_shot_prompt = f"""Analyze product reviews in this exact format:

Review: "Amazing phone, great camera, but too expensive."
Sentiment: MIXED
Pros: great camera
Cons: too expensive
Rating: 3.5/5

Review: "Perfect headphones, noise cancellation is incredible, very comfortable."
Sentiment: POSITIVE
Pros: noise cancellation, comfort
Cons: none mentioned
Rating: 5/5

Review: \"{test_review}\"
Sentiment:"""

print("=== Few-Shot ===")
print(chat(few_shot_prompt, temperature=0))

=== Few-Shot ===


Review: "The laptop arrived quickly but the packaging was damaged.
The device itself works fine â€” fast processor, beautiful display.
However, the battery barely lasts 3 hours which is disappointing for the price.
Customer support was helpful when I reported the packaging issue."
Sentiment: MIXED
Pros: quick delivery, fast processor, beautiful display, helpful customer support
Cons: damaged packaging, short battery life (barely 3 hours), disappointing for the price
Rating: 3/5


In [ ]:
# استراتژی ۳: Chain-of-Thought
cot_prompt = f"""Analyze this product review step by step:
1. First, identify all positive points mentioned
2. Then, identify all negative points mentioned
3. Consider the overall tone and weight of positive vs negative
4. Assign a final sentiment and rating

Review: \"{test_review}\"

Step-by-step analysis:"""

print("=== Chain-of-Thought ===")
print(chat(cot_prompt, temperature=0))

=== Chain-of-Thought ===


1. Positive points
- Fast delivery ("arrived quickly")
- Device performance ("works fine — fast processor")
- Display quality ("beautiful display")
- Helpful customer support for packaging issue

2. Negative points
- Packaging arrived damaged
- Short battery life ("barely lasts 3 hours"), described as disappointing given the price

3. Overall tone and weight
- Mixed/neutral leaning negative: several strong positives (performance, display, support, delivery) but a significant negative (poor battery life) that impacts value for money; packaging damage is mitigated by helpful support.

4. Final sentiment and rating
- Sentiment: Mixed/Somewhat Negative
- Rating: 3 out of 5


## خلاصه

| استراتژی | کاربرد |
|----------|------------|
| **Zero-shot** | وظایف ساده و خوش‌تعریف |
| **Few-shot** | وقتی به قالب خروجی یا رفتار خاصی نیاز دارید |
| **Chain-of-Thought** | استدلال پیچیده، ریاضی، تحلیل چندمرحله‌ای |

**نکات کلیدی:**
- در پرامپت‌های خود دقیق و صریح باشید
- پرامپت‌های سیستم رفتار را شکل می‌دهند؛ پرامپت‌های کاربر وظیفه را ارائه می‌دهند
- مثال‌های few-shot برای کنترل قالب قدرتمند هستند
- CoT دقت استدلال را به طور قابل توجهی بهبود می‌بخشد

**جلسه بعدی:** گرفتن خروجی‌های ساختاریافته (JSON) از LLMها!